# Model Training and Evaluation

This notebook demonstrates the models trained for Classification, Regression, and Clustering. It is imported directly by `app.py`.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_squared_error, r2_score, silhouette_score

### 1. Data Preparation for Models

In [ ]:
def prepare_data_for_training(df, target, problem_type):
    X = df.drop(columns=[target]) if target in df.columns else df.copy()
    X = X.select_dtypes(include=[np.number])
    X = X.dropna(axis=1, how='all')
    
    if X.empty:
        raise ValueError("Error: No valid numeric features found.")
    X = X.fillna(X.mean()).fillna(0)
    y = df[target] if target in df.columns else None
    
    if y is not None:
        mask = y.notnull()
        X = X[mask]
        y = y[mask]
    if len(X) < 2:
        raise ValueError("Error: Not enough data points.")
    if problem_type == 'regression' and target in df.columns and df[target].dtype == 'object':
        raise ValueError(f"Error: Target '{target}' is text but being used for Regression.")
    if problem_type == 'classification' and y is not None:
        if len(y.unique()) > 20 and pd.api.types.is_numeric_dtype(y):
            raise ValueError(f"Error: Target '{target}' has too many unique numerical values ({len(y.unique())}) for Classification. Try 'Regression' instead!")
        
    X = X.astype(float)
    return X, y

### 2. General Training and Evaluation Logic

In [ ]:
def train_and_evaluate_model(X, y, problem_type, model_name, save_path='models/last_model.joblib'):
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) if y is not None else (X, X, None, None)
    except Exception as e:
        raise ValueError(f"Error during data split: {str(e)}")
        
    model = None
    results = {}
    
    try:
        if problem_type == 'classification':
            if model_name == 'logistic': model = LogisticRegression(max_iter=1000)
            elif model_name == 'knn': model = KNeighborsClassifier()
            elif model_name == 'svm': model = SVC()
            else: raise ValueError("Invalid classification model")
            
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            results['accuracy'] = accuracy_score(y_test, y_pred)
            results['cm'] = confusion_matrix(y_test, y_pred).tolist()
            report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
            results['report'] = {k.replace(' ', '_'): v for k, v in report.items()}
            
            class_counts = y.value_counts()
            cv_folds = min(5, class_counts.min())
            if cv_folds >= 2:
                results['cv_mean'] = cross_val_score(model, X, y, cv=cv_folds).mean()
            else:
                results['cv_mean'] = results['accuracy']

        elif problem_type == 'regression':
            if model_name == 'linear': model = LinearRegression()
            elif model_name == 'rf': model = RandomForestRegressor()
            else: raise ValueError("Invalid regression model")
            
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            results['rmse'] = np.sqrt(mean_squared_error(y_test, y_pred))
            results['r2'] = r2_score(y_test, y_pred)
            results['cv_mean'] = cross_val_score(model, X, y, cv=min(5, len(X)), scoring='r2').mean()
            if hasattr(model, 'feature_importances_'):
                importance = dict(zip(X.columns, model.feature_importances_.tolist()))
                results['feature_importance'] = dict(sorted(importance.items(), key=lambda x: x[1], reverse=True))
                
        elif problem_type == 'clustering':
            if model_name == 'kmeans': model = KMeans(n_clusters=3, n_init=10)
            elif model_name == 'hierarchical': model = AgglomerativeClustering(n_clusters=3)
            else: raise ValueError("Invalid clustering model")
            
            clusters = model.fit_predict(X)
            results['silhouette'] = silhouette_score(X, clusters) if len(set(clusters)) > 1 else 0
            
    except Exception as e:
        raise ValueError(f"Error during model training: {str(e)}")
        
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    joblib.dump(model, save_path)
    return model, results